## Session 3, in class: value function iteration off the grid

Computational Macro (HWS 2026), University of Mannheim

Last week the household could only choose capital **on** the grid. Today the choice $k'$ is
continuous. For that we need two tools: a way to evaluate $v$ **between** grid points
(interpolation) and a way to maximize a continuous function (golden section search). Every gap is
marked `___` and belongs to one line of the slides, the comment next to it says what goes in. Each part ends with a small test or a number to compare: do not move on before it fits.

**The problem** is the growth model of sessions 1 and 2,

$$
v(k) = \max_{k_1 \leq k' \leq \min\{k^\alpha, k_N\}} \bigl\{ \ln(k^\alpha - k') + \beta \, \hat v(k') \bigr\},
$$

where $\hat v$ interpolates the vector $\mathbf{v}$ between the grid points. The closed form is the
benchmark: $v(k) = E + F \ln k$ and $k' = \alpha\beta k^\alpha$.

**Plan**

| Part | Content | Time |
| --- | --- | --- |
| 1 | Setup, and a closer look at last week's policy | 5 min |
| 2 | Tool 1: linear interpolation, and the cubic spline from a package | 12 min |
| 3 | Tool 2: maximizing a continuous function | 5 min |
| 4 | Value function iteration off the grid | 8 min |
| 5 | At home: Howard's improvement algorithm | |
| 6 | At home: experiments | |


**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $x_i, f_i$ | interpolation nodes and the values there | `xgrid`, `fgrid` |
| $w = \frac{x - x_i}{x_{i+1} - x_i}$ | weight on the right neighbour | `w` |
| $\hat v(k')$ | interpolated value function | `vhat` (a function) |
| $[a, b]$ | bracket of the golden section search | `a`, `b` |
| $b_i = \min\{k_i^\alpha, k_N\}$ | upper end of the bracket at state $k_i$ | `b` |
| $\mathbf{v}$, $\mathbf{v}^{\text{new}}$ | value function on the grid | `v`, `vnew` |
| $k'_i$ | policy at state $k_i$, now a real number and not an index | `kprime` |
| $\boldsymbol{\Gamma}$ | interpolation weights at the policy (part 5) | `Γ` |

Everything else is named as last week: `par`, `mpar`, `gri.k`, `util`, `v_true`, `kprime_true`, `distVF`, `iterVF`.

### 0. Packages

The same as last week, plus one new package for the cubic spline, `DataInterpolations`. Linear
interpolation and the maximization are written by hand, so that you see what a package does inside.

**If the cell below fails** with `Package DataInterpolations not found`, the package is not on your
laptop yet. Install it once, then run the cell again:

1. **Pull the course repository.** In GitHub Desktop: select the course
   repository, click "Fetch origin" and then "Pull origin". Or, in a terminal
   in the repository folder, run `git pull`.
2. **Open a terminal in VS Code.** Open the repository folder in VS Code (File,
   Open Folder), then open a terminal with Terminal, New Terminal (or
   ``Ctrl+` ``). The terminal starts in the repository folder, the one that
   contains `Julia/` and `Python/`.
3. **Move to the Julia folder.** Type

   ```
   cd Julia
   ```

   and press Enter. This works in PowerShell (Windows) and in the terminal on
   macOS and Linux. Check with `ls`: you should see `Project.toml` and
   `Manifest.toml`.
4. **Start Julia** in this folder by typing `julia` and pressing Enter. The
   prompt changes to `julia>`.
5. **Update the packages.** Press `]` to enter the package mode, the prompt
   changes to `(@v1.10) pkg>` (or your Julia version). Then run

   ```
   activate .
   instantiate
   ```

   `activate .` selects the course environment in this folder (the prompt now
   reads `(Julia) pkg>`), `instantiate` installs every package listed in
   `Manifest.toml` that is missing on your laptop. This downloads and compiles
   the new packages and can take a few minutes.
6. **Check.** Press backspace to leave the package mode, then type
   `using DataInterpolations`. If no error appears, you are done. Leave Julia
   with `exit()`.
7. **Restart the notebook kernel** if a notebook was open in VS Code while you
   did this (the restart button at the top of the notebook), so that it sees the
   new package.

If you started Julia somewhere else, you do not need to quit: in Julia, type
`cd("path/to/the/repository/Julia")` (with your path, forward slashes also work
on Windows), check with `pwd()`, and continue with step 5.

In [ ]:
using Plots
using Printf
using LinearAlgebra
using DataInterpolations                                 # new today: the cubic spline comes from a package

### 1. Setup: parameters, grid, utility, closed form

Identical to last week, with two changes in the numerical parameters: the grid has only
$N = 50$ points, and `interp` selects how $v$ is evaluated between grid points.

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3          # capital share
    β::Float64 = 0.96         # discount factor
end

Base.@kwdef struct NumericalParameters
    nk::Int = 50              # number of points on the capital grid, N
    kmin_rel::Float64 = 0.1   # lower grid bound, relative to k*
    kmax_rel::Float64 = 2.0   # upper grid bound, relative to k*
    spacing::Symbol = :linear # :linear or :log
    interp::Symbol = :spline  # :linear or :spline, how v is evaluated between grid points
    crit::Float64 = 1e-6      # convergence tolerance, ε
    maxiter::Int = 5000       # iteration cap, the safety net
end

par = EconomicParameters()
mpar = NumericalParameters()
println("Economic parameters:  ", par)
println("Numerical parameters: ", mpar)

The grid, the utility function, and the closed form. The grid is built by a function today,
`make_grid`, because the experiments at the end build several grids. Its two cases are those of
last week.

In [ ]:
function make_grid(par, mpar)
    kstar = (par.α * par.β)^(1 / (1 - par.α))
    kmin, kmax = mpar.kmin_rel * kstar, mpar.kmax_rel * kstar
    # two cases. If spacing is :linear, return hands back the equidistant grid and the function stops here.
    # Only if spacing is not :linear does the code reach the last line, which returns the log spaced grid
    if mpar.spacing == :linear
        return (k = collect(range(kmin, kmax, length = mpar.nk)),)             # equidistant in k
    end
    return (k = exp.(range(log(kmin), log(kmax), length = mpar.nk)),)          # equidistant in ln k
end

util(c) = log(c)

function closed_form(par, k)
    F_star = par.α / (1 - par.α * par.β)
    E_star = (log(1 - par.α * par.β) + par.β * F_star * log(par.α * par.β)) / (1 - par.β)
    return E_star .+ F_star .* log.(k), par.α * par.β .* k .^ par.α   # value and policy
end

kstar = (par.α * par.β)^(1 / (1 - par.α))
gri = make_grid(par, mpar)
v_true, kprime_true = closed_form(par, gri.k)
@printf("k* = %.4f, %d grid points from %.4f to %.4f\n", kstar, mpar.nk, gri.k[1], gri.k[end])

### 1. Exercise: a closer look at last week's policy

Last week's figure showed the grid policy on top of the closed form, and the two lines looked
identical. The function below is last week's loop in compact form, with the names of last week.
Solve on the grid with $N = 200$ and **zoom in** on the policy around the steady state,
$k \in [0.9\,k^*, 1.1\,k^*]$.

What do you see, and why can it not be otherwise?

In [ ]:
function solve_on_grid(par, k; crit = 1e-6, maxiter = 5000)
    # last week's loop in compact form: the reward matrix U once, then the row-wise maximum of M until the distance is small
    meshes = (
        k      = [k_i for k_i in k, kP in k],             # k_i in row i, the same in every column
        kprime = [kP  for k_i in k, kP in k],             # k'_j in column j, the same in every row
    )
    C = meshes.k .^ par.α .- meshes.kprime                # consumption for every (state, choice) pair
    U = fill(-Inf, size(C))                               # -Inf marks the infeasible pairs
    U[C .> 0] = util.(C[C .> 0])
    v, g, distVF, iterVF = zeros(length(k)), ones(Int, length(k)), [Inf], 0
    while distVF[end] > crit && iterVF < maxiter
        M = U .+ par.β .* v'                              # M(v) = U + β 1 v'
        vals, idx = findmax(M, dims = 2)                  # maximum along each row ...
        vnew = vec(vals)
        g = vec(getindex.(idx, 2))                        # ... and the column where it sits
        push!(distVF, maximum(abs.(vnew .- v)))
        v = vnew
        iterVF += 1
    end
    return v, k[g]                                        # value, and the policy in levels k_{g_i}
end

k_on = make_grid(par, NumericalParameters(nk = 200)).k   # last week's grid: N = 200, everything else at its default
v_on, kprime_on = solve_on_grid(par, k_on)
kprime_cf = closed_form(par, k_on)[2]                    # the closed form policy on the same grid, for comparison
@printf("on the grid, N = 200: largest policy error %.2e\n", maximum(abs.(kprime_on .- kprime_cf)))

In [ ]:
# which grid points lie within 10 percent of the steady state? zoom is a vector of true and false, one entry per grid point
zoom = ___                                               # true where k_on lies between 0.9 * kstar and 1.1 * kstar: two comparisons in brackets with .> and .<, joined by .&
@printf("%d of %d grid points lie in the zoom window\n", count(zoom), length(k_on))

In [ ]:
# left panel: the policy on the whole grid, against the closed form
p1 = plot(k_on, kprime_on, label = "VFI on the grid", legend = :topleft,
          title = "The whole grid", xlabel = "capital today k", ylabel = "capital tomorrow k'")
plot!(p1, k_on, kprime_cf, color = :black, linestyle = :dash, label = "closed form")

# right panel: only the grid points in the zoom window. k_on[zoom] keeps the entries where zoom is true.
# seriestype = :stepmid draws a step plot, which shows the policy as it is: constant, then a jump
p2 = plot(k_on[zoom], kprime_on[zoom], seriestype = :stepmid, label = "VFI on the grid", legend = :topleft,
          title = "Zoom around k*", xlabel = "capital today k")
plot!(p2, k_on[zoom], kprime_cf[zoom], color = :black, linestyle = :dash, label = "closed form")

# both panels side by side
plot(p1, p2, layout = (1, 2), size = (1100, 400), margin = 5Plots.mm)

**What you should see.** The policy is a staircase. It can only take the $N$ values on the grid,
so it jumps by one grid spacing at a time, and the error can never be smaller than half a grid
spacing. A finer grid makes the steps smaller, at the cost of $N^2$ entries in $\mathbf{M}$.
The alternative: let the household choose **between** the grid points.

### 2. Tool 1: linear interpolation

We know $f_i = f(x_i)$ at the nodes and want $\hat f(x)$ at any $x$. Find the interval
$[x_i, x_{i+1}]$ that contains $x$, and connect the two neighbours by a straight line:

$$
\hat f(x) = (1 - w)\, f_i + w\, f_{i+1}, \qquad w = \frac{x - x_i}{x_{i+1} - x_i} .
$$

Two steps in code: **locate** the left neighbour $i$, then compute the **weight** $w$.

In [ ]:
function lin_interp(xgrid, fgrid, x)
    # step 1, locate: index i of the left neighbour, x_i <= x < x_{i+1}.
    # searchsortedlast returns the index of the last node that is <= x.
    # clamp keeps i between the first and the second to last node, so that i + 1 always exists
    i = clamp(searchsortedlast(xgrid, x), 1, length(xgrid) - 1)
    # step 2, weigh: w is the distance from the left neighbour as a share of the interval, the weight on the right neighbour
    w = ___                                              # weigh: the distance of x from the left neighbour xgrid[i], divided by the length of the interval
    return ___                                           # the formula of the slides: weight 1 - w on the left value fgrid[i], weight w on the right value
end

# test: halfway between the nodes 1 and 2 the interpolant is halfway between 10 and 20
lin_interp([0.0, 1.0, 2.0], [0.0, 10.0, 20.0], 1.5)

### 2. Tool 1, continued: the natural cubic spline, from a package

On every interval $[x_i, x_{i+1}]$ the spline is its own cubic polynomial with **four**
coefficients. The slides give the four conditions that pin them down: the piece goes through
$f_i$ and through $f_{i+1}$, and at the nodes its first and its second derivative equal those of
the neighbouring piece. At the two ends of the grid the natural spline sets the second derivative
to zero. Solving this linear system is what the package does for us:

* Julia: `CubicSpline(fgrid, xgrid)` from `DataInterpolations`, **values first, then nodes**,
* Python: `CubicSpline(xgrid, fgrid, bc_type="natural")` from `scipy.interpolate`.

Both return an object that is called like a function. `make_interpolant` hides the difference
between the two methods: whichever method you pick, it returns a function of $x$. The rest of the
notebook only calls `make_interpolant`.

In [ ]:
function make_interpolant(xgrid, fgrid, method)
    # returns a FUNCTION x -> interpolated value, for either method
    if method == :linear
        return x -> lin_interp(xgrid, fgrid, x)          # our own function from above. x -> ... is a function of x that remembers xgrid and fgrid
    end
    spl = CubicSpline(fgrid, xgrid)                      # the package solves for the coefficients here, once. Values first, then nodes
    return x -> spl(x)                                   # a function of x, as in the linear case
end

### 2. Exercise: replicate the figure from the slides

Interpolate two functions on $[-3, 3]$ from **9 nodes** only, once linearly and once with the
spline, and compare both with the truth on a fine grid:

* a wave, $f(x) = \sin(2x)$,
* a function with a kink, $f(x) = \max\{x, 0\}$.

Where does the spline win, and where does it do something the true function does not?

In [ ]:
nodes = collect(range(-3, 3, length = 9))                # the 9 points where we know the function
xfine = collect(range(-3, 3, length = 400))              # a fine grid, to see what happens between the nodes

wave(x) = sin(2x)                                        # smooth everywhere
kink(x) = max(x, 0.0)                                    # a kink at x = 0, as a borrowing limit will cause

In [ ]:
# the wave: build both interpolants from the 9 values at the nodes, and evaluate them on the fine grid
wave_lin = ___                                                      # as in the line below, with the method :linear
wave_spl = make_interpolant(nodes, wave.(nodes), :spline).(xfine)

# the kink: the same two lines
kink_lin = make_interpolant(nodes, kink.(nodes), :linear).(xfine)
kink_spl = make_interpolant(nodes, kink.(nodes), :spline).(xfine)

# the largest error on the fine grid, for each function and each method
@printf("wave: largest error linear %.3f, spline %.3f\n", maximum(abs.(wave_lin .- wave.(xfine))), maximum(abs.(wave_spl .- wave.(xfine))))
@printf("kink: largest error linear %.3f, spline %.3f\n", maximum(abs.(kink_lin .- kink.(xfine))), maximum(abs.(kink_spl .- kink.(xfine))))

In [ ]:
# left panel: the wave. Dashed the truth, then the two interpolants, black dots the 9 nodes
p1 = plot(xfine, wave.(xfine), color = :black, linestyle = :dash, label = "f(x)",
          title = "A wave, sin(2x)", xlabel = "x", legend = :topleft)
plot!(p1, xfine, wave_lin, label = "linear")
plot!(p1, xfine, wave_spl, label = "spline")
scatter!(p1, nodes, wave.(nodes), color = :black, markersize = 3, label = "")

# right panel: the kink, the same four lines
p2 = plot(xfine, kink.(xfine), color = :black, linestyle = :dash, label = "f(x)",
          title = "A kink, max(x, 0)", xlabel = "x", legend = :topleft)
plot!(p2, xfine, kink_lin, label = "linear")
plot!(p2, xfine, kink_spl, label = "spline")
scatter!(p2, nodes, kink.(nodes), color = :black, markersize = 3, label = "")

# both panels side by side
plot(p1, p2, layout = (1, 2), size = (1100, 400), margin = 5Plots.mm)

**What you should see.** On the smooth wave the spline is far closer than the straight lines,
with the same 9 numbers stored. At the kink the spline overshoots and wiggles, because it insists
on a continuous second derivative that the true function does not have. Keep this in mind for
the borrowing limit in the coming weeks. Our value function $E + F \ln k$ is smooth, so today the
spline is the better tool.

### 2. Exercise: what the spline promises at the nodes

The package did not tell us how it computed the spline. Check the conditions of the slides
yourself, on the wave:

1. the spline goes **through** the 9 points,
2. its **first derivative** is continuous, also at the nodes, where one piece hands over to the next,
3. its **second derivative** is continuous as well, and zero at both ends (natural spline).

Derivatives of a `DataInterpolations` spline: `DataInterpolations.derivative(spl, x, 1)` is the
first derivative at the number `x`, with a `2` at the end the second. For comparison, the slope of
linear interpolation is one constant per interval.

In [ ]:
spl = CubicSpline(wave.(nodes), nodes)                     # the spline of the wave, kept as an object this time, because we need its derivatives

# condition 1: at the nodes the spline returns the data
@printf("largest gap between spline and data at the nodes: %.1e\n", maximum(abs.(spl.(nodes) .- wave.(nodes))))

# conditions 2 and 3: the derivatives of the spline on the fine grid
d1 = ___                                                         # first derivative of spl at every point of xfine: as in the line below, with order 1
d2 = [DataInterpolations.derivative(spl, x, 2) for x in xfine]   # second derivative
d1_lin = diff(wave.(nodes)) ./ diff(nodes)                        # slope of linear interpolation: (f_{i+1} - f_i) / (x_{i+1} - x_i), one per interval
@printf("second derivative at the two ends: %.1e and %.1e\n", d2[1], d2[end])

In [ ]:
# left panel: the spline, one colour per piece. Each piece is its own cubic, and neighbouring pieces meet at the nodes
p1 = plot(title = "The spline, piece by piece", xlabel = "x", legend = false)
for i in 1:length(nodes)-1
    x_piece = range(nodes[i], nodes[i+1], length = 50)   # the interval [x_i, x_{i+1}]
    plot!(p1, x_piece, spl.(x_piece), linewidth = 2)
end
scatter!(p1, nodes, wave.(nodes), color = :black, markersize = 3)

# middle panel: the slope. The slope of the spline is continuous, the slope of linear interpolation jumps at every node
p2 = plot(xfine, d1, label = "spline", title = "First derivative", xlabel = "x", legend = :bottomleft)
plot!(p2, nodes, [d1_lin; d1_lin[end]], seriestype = :steppost, label = "linear")   # constant on each interval, a jump at each node
scatter!(p2, nodes, [DataInterpolations.derivative(spl, x, 1) for x in nodes], color = :black, markersize = 3, label = "")

# right panel: the curvature. A straight line on each interval, continuous at the nodes, zero at both ends
p3 = plot(xfine, d2, label = "", title = "Second derivative", xlabel = "x")
scatter!(p3, nodes, [DataInterpolations.derivative(spl, x, 2) for x in nodes], color = :black, markersize = 3, label = "")
hline!(p3, [0.0], color = :gray, linewidth = 0.8, label = "")

# the three panels side by side
plot(p1, p2, p3, layout = (1, 3), size = (1400, 400), margin = 5Plots.mm)

**What you should see.** The gap at the nodes is zero up to rounding: the spline goes through
every point. On the left, each colour is one cubic, and the pieces hand over at the nodes without a
visible corner. In the middle, the slope of the spline is a smooth curve, while the slope of linear
interpolation jumps at every node: those jumps are the kinks of the straight lines. On the right,
the second derivative is a straight line on each interval (the second derivative of a cubic is
linear), the lines meet at the nodes, and both ends sit at zero. Four conditions per piece, four
coefficients per piece.

### 3. Tool 2: maximizing a continuous function

Last week the maximum was a search over one row of a table. Now the choice is continuous, and the
computer needs (i) a **bracket** $[a, b]$ to search in and (ii) a rule that shrinks the bracket.
Golden section search evaluates the function at two interior points $c < d$ and throws away the
part of the bracket that cannot contain the maximum:

* if $f(c) > f(d)$, the maximum is in $[a, d]$,
* otherwise it is in $[c, b]$.

The interior points sit at the golden ratio $\varphi = (\sqrt 5 - 1)/2 \approx 0.618$, so that one
of them can be reused and every step costs **one** new function evaluation. The bracket shrinks by
the factor $\varphi$ per step. It needs no derivative, only a function with a single peak.

In [ ]:
function golden_max(f, a, b; tol = 1e-10)
    φ = (sqrt(5) - 1) / 2                                 # the golden ratio, about 0.618
    c, d = b - φ * (b - a), a + φ * (b - a)               # two interior points, c < d
    fc, fd = f(c), f(d)
    while b - a > tol                                     # until the bracket is narrower than tol
        if fc > fd                                        # maximum in [a, d]: d becomes the new upper end
            b, d, fd = d, c, fc                           # three updates at once: d is the new upper end b, the old c becomes the new d, and its value fc comes along
            c = b - φ * (b - a)                           # the old c is reused as d, only c is new
            fc = f(c)                                     # the one new function evaluation of this step
        else                                              # maximum in [c, b]: c becomes the new lower end
            a, c, fc = ___                                   # the mirror image of the line b, d, fd = d, c, fc above: c is the new lower end, the old d becomes the new c, its function value comes along
            d = a + φ * (b - a)                           # the old d is reused as c, only d is new
            fd = f(d)
        end
    end
    x = (a + b) / 2
    return x, f(x)                                        # the maximizer and the maximum
end

# test: ln(x) + ln(1 - x) peaks at x = 0.5
golden_max(x -> log(x) + log(1 - x), 0.0, 1.0)

### 4. Value function iteration off the grid: one Bellman step

The state is still on the grid, the choice is not. Line 4 and line 5 of the algorithm on the slides:

* line 4: interpolate $\mathbf{v}$, which gives the function $\hat v$,
* line 5, for every state $k_i$: resources are $y_i = k_i^\alpha$, the right hand side is a
  **function** of the choice, $k' \mapsto \ln(y_i - k') + \beta\, \hat v(k')$, the bracket is
  $[k_1, b_i]$ with $b_i = \min\{y_i, k_N\}$ (consumption must stay positive, and we do not want to
  evaluate $\hat v$ outside the grid), and golden section search returns the maximizer $k'_i$ and
  the maximum $v^{\text{new}}_i$.

Compare with last week: the matrix $\mathbf{M}$ and the row-wise maximum are gone, a loop over
states with one search each has taken their place.

In [ ]:
function vfi_update(v, par, mpar, gri)
    vhat = make_interpolant(gri.k, v, mpar.interp)       # line 4: v between the grid points, built once per step
    vnew, kprime = zeros(mpar.nk), zeros(mpar.nk)
    for i in 1:mpar.nk                                   # line 5: loop over the states k_i
        y = gri.k[i]^par.α                               # resources at k_i
        rhs = kp -> ___                                  # right hand side as a function of the choice kp: utility of consumption y - kp, plus β times vhat at kp
        b = min(___, ___)                                # upper end b_i of the bracket: the smaller of resources (minus 1e-10, so that c > 0) and the last grid point
        kprime[i], vnew[i] = ___                            # golden_max(function, lower end, upper end): search rhs between the first grid point and b
    end
    return vnew, kprime
end

### 4. The loop

Unchanged from last week, except that one line calls `vfi_update` where the matrix $\mathbf{M}$
used to be. The theory is unchanged as well: the presentation today showed that the Bellman operator
is a contraction, so the distance still falls by the factor $\beta$ per iteration.

In [ ]:
timer = time()
v = zeros(mpar.nk)                                       # line 2: the guess v_0 = 0
kprime = zeros(mpar.nk)
distVF, iterVF = [Inf], 0                                # the distances d_n, one per iteration, and the counter
while distVF[end] > mpar.crit && iterVF < mpar.maxiter   # line 3
    global v, kprime, iterVF                             # the loop sits at the top level of the notebook
    vnew, kprime = ___                                   # lines 4 and 5: one Bellman step, off the grid. Call vfi_update with the current guess v
    dd = maximum(abs.(vnew .- v))                        # line 6: the distance ‖v_new - v‖_∞
    v = vnew                                             #         and the update
    iterVF += 1
    push!(distVF, dd)
end
time_vfi = time() - timer
if distVF[end] > mpar.crit
    error("not converged: the iteration cap was hit")
end
@printf("iterations: %d, final distance: %.3e, last ratio: %.4f, time: %.1f s\n", iterVF, distVF[end], distVF[end] / distVF[end-1], time_vfi)

### 4. Checks against the closed form

The same checks as last week. Compare with the grid solution of part 1, which used four times as
many grid points.

In [ ]:
@printf("policy at the grid bounds: %d states at k_1, %d states at k_N\n", count(kprime .<= gri.k[1] + 1e-8), count(kprime .>= gri.k[end] - 1e-8))
@printf("largest policy error off the grid, N = %d (%s): %.2e\n", mpar.nk, mpar.interp, maximum(abs.(kprime .- kprime_true)))
@printf("largest policy error on the grid,  N = 200:          %.2e\n", maximum(abs.(kprime_on .- kprime_cf)))
@printf("largest value error: %.2e   (certificate β/(1-β) d: %.2e)\n", maximum(abs.(v .- v_true)), par.β / (1 - par.β) * distVF[end])

The zoom of part 1 again, now with the policy off the grid. To see the policy between the states
as well, it is interpolated like any other function on the grid.

In [ ]:
kfine = collect(range(0.9 * kstar, 1.1 * kstar, length = 300))   # a fine grid in the zoom window
kprime_fine = make_interpolant(gri.k, kprime, mpar.interp).(kfine)   # the policy off the grid, interpolated between the states

plot(k_on[zoom], kprime_on[zoom], seriestype = :stepmid, label = "on the grid, N = 200", legend = :topleft,
     title = "Policy, zoom around k*", xlabel = "capital today k", ylabel = "capital tomorrow k'", size = (650, 400))
plot!(kfine, kprime_fine, label = "off the grid, N = $(mpar.nk)")
plot!(kfine, closed_form(par, kfine)[2], color = :black, linestyle = :dash, label = "closed form")

**What you should find.** 337 iterations as last week, since the contraction rate is $\beta$
whatever the method. With 50 grid points and the spline the policy error is about $10^{-6}$, a
thousand times smaller than on the grid with 200 points. The value error is $2 \cdot 10^{-5}$ and
no longer comes from the grid: it is the stopping error, bounded by the certificate. The price is
the run time, 337 iterations times 50 searches times about 50 function evaluations. Part 5, at
home, removes most of the iterations.

### 5. At home: Howard's improvement algorithm

Not covered in class. Work through it before next week, the session on policy function iteration
builds on it.

A Bellman step uses the new policy for **one** period and then continues with the old value.
Howard's idea: use the new policy **forever**. The value of following the policy $k'_i$ forever
solves

$$
v_i = u(k_i^\alpha - k'_i) + \beta\, \hat v(k'_i) \qquad \text{for all } i .
$$

Interpolation is **linear in the values** $\mathbf{v}$: $\hat v(k'_i) = \sum_j \Gamma_{ij} v_j$, where row
$i$ of $\boldsymbol{\Gamma}$ holds the interpolation weights at $k'_i$. With linear interpolation the row has two
entries, $1 - w$ on the left neighbour and $w$ on the right one, a lottery over the two
neighbours. The system is therefore linear and can be solved in one step,

$$
\mathbf{v} = \mathbf{u} + \beta \boldsymbol{\Gamma} \mathbf{v}
\quad \Longleftrightarrow \quad
\mathbf{v} = (\mathbf{I} - \beta \boldsymbol{\Gamma})^{-1} \mathbf{u} .
$$

Column $j$ of $\boldsymbol{\Gamma}$ is what the interpolant returns at the policy when $\mathbf{v}$ is the $j$-th unit
vector. That builds $\boldsymbol{\Gamma}$ for the linear interpolant and the spline alike. With the spline the rows
still sum to one, but they have more than two entries and some are negative.

In [ ]:
function howard_step(kprime, par, mpar, gri)
    eye = Matrix(1.0I, mpar.nk, mpar.nk)                 # the identity matrix I: column j is the j-th unit vector, 1 at grid point j and 0 elsewhere
    Γ = zeros(mpar.nk, mpar.nk)
    # column j of Γ: pretend v is the j-th unit vector, interpolate it, and evaluate it at the policy.
    # The result is the weight that v_j receives in vhat(k'_i), for every state i
    for j in 1:mpar.nk
        Γ[:, j] = ___                                    # as in part 2: an interpolant with nodes gri.k and values eye[:, j], evaluated at kprime with a dot
    end
    u = ___                                              # payoff under the policy, one number per state: utility of output minus kprime, all with dots
    return ___                                           # solve (I - β Γ) v = u for v with the backslash: matrix \ right hand side
end

The loop: find the policy with one Bellman step, then replace the value by the value of following
that policy forever. Stop when the value no longer moves.

In [ ]:
timer = time()
v_pfi = zeros(mpar.nk)
kprime_pfi = zeros(mpar.nk)
distPF, iterPF = [Inf], 0
while distPF[end] > mpar.crit && iterPF < mpar.maxiter
    global v_pfi, kprime_pfi, iterPF
    _, kprime_pfi = vfi_update(v_pfi, par, mpar, gri)    # policy improvement: the maximization
    vnew = howard_step(kprime_pfi, par, mpar, gri)       # policy evaluation: the linear system
    dd = maximum(abs.(vnew .- v_pfi))
    v_pfi = vnew
    iterPF += 1
    push!(distPF, dd)
end
time_pfi = time() - timer
@printf("VFI:    %4d iterations, %6.2f s, policy error %.2e, value error %.2e\n", iterVF, time_vfi, maximum(abs.(kprime .- kprime_true)), maximum(abs.(v .- v_true)))
@printf("Howard: %4d iterations, %6.2f s, policy error %.2e, value error %.2e\n", iterPF, time_pfi, maximum(abs.(kprime_pfi .- kprime_true)), maximum(abs.(v_pfi .- v_true)))

In [ ]:
plot(1:iterVF, distVF[2:end], yaxis = :log10, linewidth = 2, label = "VFI",
     title = "Distance between iterates", xlabel = "iteration n", size = (650, 400))
plot!(1:iterPF, max.(distPF[2:end], 1e-16), marker = :circle, label = "Howard")   # a distance of exactly zero has no logarithm
hline!([mpar.crit], color = :gray, linestyle = :dot, label = "tolerance")

**What you should find.** Howard's algorithm needs 5 iterations where VFI needs 337. The policy is
the same. The value is **more** accurate, because the linear system removes the stopping error
altogether: what is left is the interpolation error alone. The distance no longer falls at the
rate $\beta$, it collapses, which is the behaviour of a Newton method. Next week's session takes
this up under the name policy function iteration.

### 6. At home: experiments

`solve_off_grid` wraps parts 4 and 5 in one function. It uses Howard's algorithm because it is
fast. Change one field of `mpar` at a time and compare.

| Try | Question |
| --- | --- |
| `interp` linear against spline, at $N = 50$ | how much does linear interpolation gain over the grid search? Why so little? |
| `nk` 10, 20, 30, 50 with the spline | how few grid points are enough to beat the grid search with $N = 2000$ (policy error $10^{-4}$)? |
| `spacing` log with the spline | does log spacing help now? |

In [ ]:
function solve_off_grid(par, mpar)
    gri = make_grid(par, mpar)
    v, kprime, d, n = zeros(mpar.nk), zeros(mpar.nk), Inf, 0
    while d > mpar.crit && n < mpar.maxiter
        _, kprime = vfi_update(v, par, mpar, gri)
        vnew = howard_step(kprime, par, mpar, gri)
        d = maximum(abs.(vnew .- v))
        v = vnew
        n += 1
    end
    v_true, kprime_true = closed_form(par, gri.k)
    return n, maximum(abs.(kprime .- kprime_true)), maximum(abs.(v .- v_true))
end

experiments = [
    "linear, N = 50"      => NumericalParameters(interp = :linear),
    "linear, N = 200"     => NumericalParameters(interp = :linear, nk = 200),
    "spline, N = 10"      => NumericalParameters(nk = 10),
    "spline, N = 20"      => NumericalParameters(nk = 20),
    "spline, N = 30"      => NumericalParameters(nk = 30),
    "spline, N = 50"      => NumericalParameters(),
    "spline, N = 50, log" => NumericalParameters(spacing = :log),
]
@printf("%-22s %10s %13s %12s\n", "run", "iterations", "policy error", "value error")
for (label, m) in experiments
    n, pol_err, val_err = solve_off_grid(par, m)
    @printf("%-22s %10d %13.1e %12.1e\n", label, n, pol_err, val_err)
end

Note your numbers here. The answers are in the solution notebook, which is posted after the session.

### In practice

The spline came from a package today, linear interpolation and the maximization were written by
hand. In your own work use a tested package for the maximization as well: `Optim.jl`, with
`optimize(f, a, b)` for a function of one variable on the bracket $[a, b]$. It minimizes, so hand
it the **negative** of the right hand side. `Optim.jl` is not in the course environment yet, we add
packages to the `Project.toml` when we first need them.